# Inicializando

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType, DateType, IntegerType
from pyspark.sql.functions import col, trim, regexp_replace, length, isnull

# Lendo dados da Tabela Bronze

In [0]:
df = spark.table('workspace.bronze.erp_px_cat_g1v2')
df.display()

# Remoção de espaços desnecessários de colunas string

In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))

df.display()
  

# Normalizando coluna ID para bater prd_key da tabela 'crm_prd_info'

In [0]:
df = (
    df.withColumn('ID', regexp_replace(trim(col('ID')), '_', '-'))
)

df.display()

# Transformando campo flag 'MAINTENANCE' em boolean

In [0]:
df = (
    df.withColumn(
        'MAINTENANCE',
        F.when(F.upper(trim(col("MAINTENANCE"))) == 'YES', F.lit(True))
        .when(F.upper(trim(col("MAINTENANCE"))) == 'NO', F.lit(False))
        .otherwise(None)
    )
)

df.display()

# Renomeando Colunas

In [0]:
RENAME_MAP = {
    'ID': 'category_id',
    'CAT': 'category',
    'SUBCAT': 'subcategory',
    'MAINTENANCE': 'maintenance_flag'
}

for old_name, new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)

df.limit(10).display()

# Criação de Tabela Silver

In [0]:
df.write.mode('overwrite').format('delta').saveAsTable('workspace.silver.erp_product_category')

# Verificação de Dados na Tabela Final

In [0]:
%sql
SELECT * FROM workspace.silver.erp_product_category LIMIT 10